In [72]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import shutil

import cv2
import numpy as np
from vitallens import VitalLens

from nbutils.hr_estimation import annotate_videos, describe

INPUT_DIR = Path("../data/input")     # tu wrzucaj filmy (.mp4/.mov/.avi/.mkv/.m4v/.webm)
OUTPUT_DIR = Path("../data/output")   # tu zapisują się filmy z nakładką HR
METHOD = "pos"                        # "pos" | "chrom" | "g"  (wszystkie lokalne, bez API)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [73]:
print("ffmpeg:", shutil.which("ffmpeg"))

videos = sorted(p for p in INPUT_DIR.iterdir()
                if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"})
print(f"Znaleziono filmów: {len(videos)}")

for p in videos:
    cap = cv2.VideoCapture(str(p))
    n, fps = cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FPS)
    w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    print(f"{p.name}: {int(w)}x{int(h)}, {fps:.1f} fps, {n / fps:.1f} s")

ffmpeg: /opt/homebrew/bin/ffmpeg
Znaleziono filmów: 1
jn.mov: 1920x1080, 30.0 fps, 30.6 s


In [74]:
vl = VitalLens(method=METHOD, estimate_rolling_vitals=True, export_to_json=False)
print("Gotowe, metoda:", METHOD)

Gotowe, metoda: pos


In [75]:
import subprocess
import base64
from IPython.display import HTML, display
from nbutils.hr_estimation import _read_frames, _series, _global, _vitals, GREEN, GREY, LIGHT


def measure_hr(vl, video_path, max_gb=1.5):
    """Liczy HR. Zwraca (hr_frames, fps, is_rolling): hr_frames[i] = HR [bpm] dla klatki i (NaN = brak)."""
    video_path = Path(video_path)
    try:
        res = vl(str(video_path))
    except Exception as e:
        print("Odczyt ze ścieżki nie wyszedł, używam OpenCV:", e)
        frames, fps_in = _read_frames(video_path, max_gb)
        res = vl(frames, fps=fps_in)
        del frames
    if not res:
        raise RuntimeError("Brak wyniku (nie wykryto twarzy?)")

    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()

    series = _series(res[0], "heart_rate")
    if series is None:                       # brak serii -> stała wartość z całego filmu (jeśli jest)
        g = _global(_vitals(res[0]), "heart_rate")
        return np.full(n, np.nan if g is None else g), fps, False
    idx = np.minimum((np.arange(n) * len(series) / n).astype(int), len(series) - 1)
    return series[idx], fps, True            # HR przypisany klatka po klatce


def render_with_hr_bar(video_path, hr_frames, fps, out_dir, tag, bar_frac=0.10, preview_width=720):
    """Dokleja czarny pasek z HR NAD filmem (osobny pas, nie na twarzy). Zwraca (pełny_film, podgląd)."""
    if not shutil.which("ffmpeg"):
        raise RuntimeError("Brak ffmpeg w PATH")
    video_path, out_dir = Path(video_path), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    full = out_dir / f"{video_path.stem}_hr_{tag}.mp4"
    preview = out_dir / f"{video_path.stem}_hr_{tag}_preview.mp4"
    raw = full.with_suffix(".raw.mp4")

    cap = cv2.VideoCapture(str(video_path))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = len(hr_frames)
    bar_h = max(48, int(h * bar_frac))
    font, s = cv2.FONT_HERSHEY_SIMPLEX, bar_h / 60.0

    # tło paska: wykres HR w czasie (stały), po środku
    bg = np.zeros((bar_h, w, 3), np.uint8)
    x0, x1, y_top, y_bot = int(0.30 * w), int(0.80 * w), int(0.18 * bar_h), int(0.82 * bar_h)
    valid = np.isfinite(hr_frames)
    xs = ys = None
    if valid.sum() > 1 and n > 1:
        lo, hi = float(np.nanmin(hr_frames)), float(np.nanmax(hr_frames))
        if hi - lo < 5:
            lo, hi = lo - 2.5, hi + 2.5
        xs = (x0 + np.arange(n) / (n - 1) * (x1 - x0)).astype(int)
        ys = np.where(valid, y_bot - (np.nan_to_num(hr_frames, nan=lo) - lo) / (hi - lo) * (y_bot - y_top), 0).astype(int)
        pts = np.stack([xs[valid], ys[valid]], axis=1).astype(np.int32).reshape(-1, 1, 2)
        cv2.polylines(bg, [pts], False, (90, 90, 90), 2, cv2.LINE_AA)

    writer = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h + bar_h))
    if not writer.isOpened():
        raise RuntimeError(f"Nie można otworzyć VideoWriter: {raw}")
    try:
        for i in range(n):
            ok, frame = cap.read()
            if not ok:
                break
            hr = hr_frames[i]
            bar = bg.copy()
            txt, col = (f"HR {hr:.0f} bpm", GREEN) if np.isfinite(hr) else ("HR -- bpm", GREY)
            (_, th), _ = cv2.getTextSize(txt, font, s, 2)
            cv2.putText(bar, txt, (int(0.02 * w), (bar_h + th) // 2), font, s, col, 2, cv2.LINE_AA)
            t_txt = f"{i / fps:5.1f} s"
            (tw2, th2), _ = cv2.getTextSize(t_txt, font, s * 0.6, 1)
            cv2.putText(bar, t_txt, (w - tw2 - int(0.02 * w), (bar_h + th2) // 2), font, s * 0.6, LIGHT, 1, cv2.LINE_AA)
            if xs is not None and valid[i]:
                cv2.circle(bar, (int(xs[i]), int(ys[i])), max(3, bar_h // 14), GREEN, -1, cv2.LINE_AA)
            writer.write(np.vstack([bar, frame]))
    finally:
        cap.release()
        writer.release()

    even = "scale=trunc(iw/2)*2:trunc(ih/2)*2"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw), "-i", str(video_path),
                    "-map", "0:v:0", "-map", "1:a:0?", "-vf", even, "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-crf", "20", "-c:a", "aac", "-shortest", "-movflags", "+faststart", str(full)], check=True)
    raw.unlink()
    # mały podgląd do wyświetlenia w notebooku (pełny film może mieć setki MB)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(full),
                    "-vf", f"scale={preview_width}:-2", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "28",
                    "-preset", "veryfast", "-c:a", "aac", "-b:a", "96k", "-movflags", "+faststart", str(preview)],
                   check=True)
    return full, preview

In [76]:
results = {}
for video in videos:
    print(f"== {video.name}")
    hr_frames, fps, rolling = measure_hr(vl, video)
    print(f"   HR: klatek {len(hr_frames)}, średnia {np.nanmean(hr_frames):.1f} bpm, seria w czasie: {rolling}")
    full, preview = render_with_hr_bar(video, hr_frames, fps, OUTPUT_DIR, METHOD)
    results[video.name] = {"hr": hr_frames, "fps": fps, "full": full, "preview": preview}
    print(f"   zapisano: {full.name} ({full.stat().st_size / 1e6:.1f} MB), podgląd: {preview.stat().st_size / 1e6:.1f} MB")

== jn.mov
Odczyt ze ścieżki nie wyszedł, używam OpenCV: Problem reading video from ../data/input/jn.mov: Unable to parse input video. There may be an issue with the video file.
  reading frames with OpenCV: 917 frames, 1920x1080 -> 984x553
   HR: klatek 917, średnia 54.4 bpm, seria w czasie: True
   zapisano: jn_hr_pos.mp4 (35.4 MB), podgląd: 1.4 MB


In [77]:
import subprocess
subprocess.run(["open", str(r["full"])])

CompletedProcess(args=['open', '../data/output/jn_hr_pos.mp4'], returncode=0)